# Topic 20a - Tidy data principles

A short practice notebook. Work through it after reading section 1 of the [main lesson](topic20_data_wrangling.ipynb).

**Tidy data** has three rules:

1. Each variable is a column.
2. Each observation is a row.
3. Each type of observational unit is its own table.

A tidy table is not "a clean table". It is a table whose *shape* matches the analysis you want to do. The same clean data can be tidy for one question and untidy for another.

In [ ]:
from pathlib import Path

import pandas as pd


def find_data_dir():
    """Locate the lesson's data folder from any sensible working directory."""
    for candidate in [Path.cwd() / "data",
                      Path.cwd() / "data_wrangling" / "data",
                      Path.cwd().parent / "data"]:
        if (candidate / "orders.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find orders.csv - check your data folder.")


DATA_DIR = find_data_dir()
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 130)
print("data folder:", DATA_DIR)

## 1. Recognising untidy shapes

Three tables follow. Each breaks a different rule. Read each one and decide which rule it breaks before running the next cell.

In [ ]:
# Table A
table_a = pd.DataFrame({
    "region": ["Greater Accra", "Ashanti", "Northern"],
    "2021": [5455692, 5440463, 2310939],
    "2022": [5580000, 5562000, 2366000],
    "2023": [5707000, 5686000, 2423000],
})

# Table B
table_b = pd.DataFrame({
    "customer": ["Kwame Mensah", "Ama Osei", "Yaw Boateng"],
    "region_segment": ["Greater Accra-Retail", "Ashanti-Wholesale", "Volta-Retail"],
    "revenue_ghs": [1240, 8600, 930],
})

# Table C
table_c = pd.DataFrame({
    "order_id": ["O1", "O2", "O3"],
    "product": ["Milo Tin 400g", "Frytol 2L", "Gari 5kg"],
    "customer_name": ["Kwame Mensah", "Kwame Mensah", "Ama Osei"],
    "customer_region": ["Greater Accra", "Greater Accra", "Ashanti"],
    "customer_joined": ["2021-03-04", "2021-03-04", "2020-11-19"],
})

for name, table in [("A", table_a), ("B", table_b), ("C", table_c)]:
    print(f"--- Table {name} ---")
    print(table.to_string(index=False))
    print()

### The answers

- **Table A** breaks rule 1. The column headers `2021`, `2022`, `2023` are *values* of a variable called `year`, not variable names. Fix with `melt()`.
- **Table B** breaks rule 1 differently: `region_segment` holds **two** variables in one column. Fix with `str.split()`.
- **Table C** breaks rule 3. It mixes two observational units - orders and customers - so Kwame Mensah's region and join date are repeated on every order he places. If he moves house you must update several rows and you will miss one. Fix by splitting it into two tables.

Rule 3 is the one that feels wrong to beginners, because Table C looks convenient. It *is* convenient to read, and that is exactly why analysis tables and storage tables are different things.

## 2. Fix Table A with `melt()`

In [ ]:
tidy_a = table_a.melt(id_vars="region", var_name="year", value_name="population")
tidy_a["year"] = tidy_a["year"].astype(int)
print(tidy_a.to_string(index=False))
print(f"\n{table_a.shape} -> {tidy_a.shape}   (3 regions x 3 years = 9 observations)")

## 3. Your turn - fix Table B

`region_segment` contains two variables joined by a hyphen. Split it into separate `region` and `segment` columns, then drop the original.

Hint: `table_b["region_segment"].str.split("-", expand=True)` returns a DataFrame with one column per piece.

In [ ]:
# YOUR CODE HERE
# split = table_b["region_segment"].str.split("-", expand=True)

## 4. Your turn - fix Table C

Split `table_c` into two tidy tables:

- `orders_tidy` with `order_id`, `product`, and `customer_name`
- `customers_tidy` with one row per customer: `customer_name`, `customer_region`, `customer_joined`

Then confirm `customers_tidy` really has one row per customer, and show that you can rebuild the original by merging them back together.

In [ ]:
# YOUR CODE HERE
# customers_tidy = table_c[["customer_name", "customer_region", "customer_joined"]].drop_duplicates()

## 5. Tidy for which question?

Load the real orders table and answer these in a markdown cell:

1. What is the grain of `orders` - what does one row represent?
2. `orders` has no `region` column. Is it therefore untidy? Explain your answer using rule 3.
3. You are asked for "total revenue by region and month". Which tables do you need, and what shape must the result be in?

In [ ]:
orders = pd.read_csv(DATA_DIR / "orders.csv")
print(orders.head(3).to_string(index=False))
print(f"\nShape: {orders.shape}")
print(f"Unique order_id values: {orders['order_id'].nunique()}")

**Your answers:**

1. *(Write your answer here.)*
2. *(Write your answer here.)*
3. *(Write your answer here.)*

---
## Where this fits

- [Back to the main lesson](topic20_data_wrangling.ipynb)
- Previous: the main lesson
- Next: [20b - Stacking with concat](topic20b_stacking_with_concat.ipynb)